# Les Dossiers de l'Inspecteur : jouer

Deux enquêtes dans le Paris des années folles, où chaque personnage est joué par le modèle de langage du projet (Qwen3-4B adapté par LoRA) :

- **Le Dernier Verre du Faisan Doré** (1928), l'affaire sur laquelle l'adaptateur a été entraîné ;
- **Dernière au Théâtre du Miroir** (1929), une intrigue que le modèle n'a jamais vue.

Choisir un GPU (*Exécution > Modifier le type d'exécution > GPU T4*), puis exécuter les trois cellules. Sans GPU, le jeu passe en **mode démo** : les personnages répondent alors avec leur fiche, sans modèle de langage.

Le dépôt est privé : le clonage demande un jeton GitHub **à soi** (compte invité sur le dépôt), rangé dans les secrets Colab sous le nom `GITHUB_TOKEN` (icône de clé, à gauche), ou collé quand la première cellule le demande. Aucun jeton Hugging Face n'est nécessaire : le modèle et l'adaptateur sont publics.

In [ ]:
import os, subprocess, sys
from pathlib import Path

DEPOT, BRANCHE = "https://github.com/Neptuniste/agent-enqueteur.git", "affaire-faisan-dore"
DANS_COLAB = "google.colab" in sys.modules
if not Path("src/jeu").exists():                        # notebook ouvert hors du dépôt (Colab)
    if not Path("agent-enqueteur").exists():
        jeton = None
        try:                                             # dépôt privé : jeton GitHub dans les secrets Colab
            from google.colab import userdata
            jeton = userdata.get("GITHUB_TOKEN")
        except Exception:
            pass
        if not jeton:
            from getpass import getpass
            jeton = getpass("Jeton GitHub (dépôt privé ; Entrée s'il est devenu public) : ").strip()
        url = DEPOT.replace("https://", f"https://{jeton}@") if jeton else DEPOT
        r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCHE, url, "agent-enqueteur"],
                           env=dict(os.environ, GIT_TERMINAL_PROMPT="0"), capture_output=True, text=True)
        if r.returncode:
            erreur = r.stderr.replace(jeton, "***") if jeton else r.stderr
            raise RuntimeError(f"Clonage impossible : {erreur.strip()}\nIl faut un compte GitHub invité sur le dépôt "
                               "et un jeton à soi, dans le secret GITHUB_TOKEN ou collé ci-dessus.")
        if jeton:                                        # le jeton ne reste pas dans .git/config
            subprocess.run(["git", "-C", "agent-enqueteur", "remote", "set-url", "origin", DEPOT], check=True)
    os.chdir("agent-enqueteur")
if DANS_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, os.getcwd())
print("Projet :", os.getcwd())

In [ ]:
import torch
from src.config import ADAPTER_REPO, MODEL_NAME
from src.llm import charger_modele

GPU = torch.cuda.is_available()
ADAPTATEUR = "adapter" if Path("adapter/adapter_config.json").exists() else ADAPTER_REPO   # ou un dépôt Hugging Face
model = tokenizer = None
if GPU:
    model, tokenizer = charger_modele(MODEL_NAME, adaptateur=ADAPTATEUR or None)
    print("Modèle :", MODEL_NAME, "| adaptateur :", ADAPTATEUR or "aucun (modèle de base)")
else:
    print("Pas de GPU : le jeu se lance en mode démo.")

In [ ]:
from src.jeu import lancer_jeu

affiche = MODEL_NAME.split("/")[-1] + (" + LoRA" if ADAPTATEUR else "")
lancer_jeu(model, tokenizer, demo=not GPU, modele_affiche=affiche, share=True)